# Installing the libraries

In [11]:
%pip install -U scikit-learn mlflow mlxtend pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


# Car insurance claim prediction (Classification)
**Student:** Moheeb Abusamra 248690 <br>
**Goal:** Predict whether a customer will file a claim or not (Claim_Flag) based on other attributes


## 1. Setup and imports, here the core libraries used are imported

In [12]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
import mlflow
import mlxtend

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
#seeds are used to ensure reproducability, every run we get the same "random" state, important for splitting
pd.set_option("display.max_columns", None) #show all columns
pd.set_option("display.max_rows", None) #show all rows
pd.set_option("display.width", 200) #200 characters max width
sns.set_theme(style="whitegrid", palette="deep") #white background and deep as color palette for all seaborn
plt.rcParams["figure.figsize"] = (10, 5) #default size for all plots to 10 x 5
#above are display settings, could be changed to ensure better output visbility


### 1.2 MLflow experiment tracking
All runs (baselines, feature-selection trials, hyperparameter searches, final models) are logged to one MLflow experiment, stored locally in `./mlruns`. Each run records its parameters, metrics and artifacts (plots, reports, models), so every result in this notebook can be traced and compared in the MLflow UI. A small test run checks that tracking works.

In [13]:
#JUST TO MAKE SURE MLFLOW IS INSTALLED, MADE A DUMMY EXPERIEMNT AND DUMMY FILLING

# Create both the mlruns folder and its trash subfolder
os.makedirs("./mlruns/.trash", exist_ok=True)

# MLflow 3.x needs this opt-in to use the local folder (file) store, as in the labs
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("file:./mlruns")

EXPERIMENT_NAME = "Car Insurance Claim - Classification"
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment  :", EXPERIMENT_NAME)

# Test run: confirms MLflow can write params and metrics
# with mlflow.start_run(run_name="00_setup_check"):
#     mlflow.log_param("random_state", RANDOM_STATE)
#     mlflow.log_param("sklearn_version", sklearn.__version__)
#     mlflow.log_metric("setup_ok", 1)

mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

Tracking URI: file:./mlruns
Experiment  : Car Insurance Claim - Classification


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.setup_ok,params.random_state,params.sklearn_version,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.runName
0,c3a5b30b54324d98b4ee6271e9b5e9f1,490695274478244813,FINISHED,file:c:/Users/Moheeb/Desktop/ML_Classification...,2026-11-01 22:27:36.776+00:00,2026-11-01 22:27:36.805+00:00,1.0,42,1.9.1,NOTEBOOK,Car_Insurance_Claim_Classification.ipynb,Moheeb,00_setup_check
1,0900375f780b4ca79ba730e470b35c04,490695274478244813,FINISHED,file:c:/Users/Moheeb/Desktop/ML_Classification...,2026-11-01 22:23:59.682+00:00,2026-11-01 22:23:59.709+00:00,1.0,42,1.9.1,NOTEBOOK,Car_Insurance_Claim_Classification.ipynb,Moheeb,00_setup_check


<h1>Loading the dataset </h1>
the dataframe named df_raw will hold the original csv

In [14]:
DATA_PATH="data/car_insurance_claim.csv" #location of the csv
df_raw= pd.read_csv(DATA_PATH) #loading the csv from the location

print ("Shape:" , df_raw.shape) #printing the shape (features and instances)
df_raw.head() # printing a sample which are the top 5 instances

Shape: (10302, 27)


,ID,KIDSDRIV,BIRTH,AGE,HOMEKIDS,YOJ,INCOME,PARENT1,HOME_VAL,MSTATUS,GENDER,EDUCATION,OCCUPATION,TRAVTIME,CAR_USE,BLUEBOOK,TIF,CAR_TYPE,RED_CAR,OLDCLAIM,CLM_FREQ,REVOKED,MVR_PTS,CLM_AMT,CAR_AGE,CLAIM_FLAG,URBANICITY
0,63581743,0,16MAR39,60.0,0,11.0,"$67,349",No,$0,z_No,M,PhD,Professional,14,Private,"$14,230",11,Minivan,yes,"$4,461",2,No,3,$0,18.0,0,Highly Urban/ Urban
1,132761049,0,21JAN56,43.0,0,11.0,"$91,449",No,"$257,252",z_No,M,z_High School,z_Blue Collar,22,Commercial,"$14,940",1,Minivan,yes,$0,0,No,0,$0,1.0,0,Highly Urban/ Urban
2,921317019,0,18NOV51,48.0,0,11.0,"$52,881",No,$0,z_No,M,Bachelors,Manager,26,Private,"$21,970",1,Van,yes,$0,0,No,2,$0,10.0,0,Highly Urban/ Urban
3,727598473,0,05MAR64,35.0,1,10.0,"$16,039",No,"$124,191",Yes,z_F,z_High School,Clerical,5,Private,"$4,010",4,z_SUV,no,"$38,690",2,No,3,$0,10.0,0,Highly Urban/ Urban
4,450221861,0,05JUN48,51.0,0,14.0,NaN,No,"$306,251",Yes,M,<High School,z_Blue Collar,32,Private,"$15,440",7,Minivan,yes,$0,0,No,0,$0,6.0,0,Highly Urban/ Urban


## 2. Dataset Description & Audit
### 2.1 Domain and motivation
**Domain:** motor (car) insurance, a risk-assessment problem. <br>
**Source:**  https://www.kaggle.com/datasets/xiaomengsun/car-insurance-claim-data <br>
**Task:** binary classification. Predict `CLAIM_FLAG` (1 = the customer filed a claim, 0 = no claim) from the customer's demographic, financial, vehicle and driving-history information.

**Why it is interesting:**
- **Real business value:** Help insurancec company in risk management as well as pricing.
- **Realistic, messy data:** Has messy data and different feature types as well as missing data and hidden data leakage column, this forces real preprocessing decision.
- **Class imbalance:** 27% of cutomers claim meanwhile the rest don't. Here we need a valid evaluation metric choice because accuracy is misleading (if you random guess you will have a 73% accuracy)
- **Interpretability matters:** in insurance, decisions must be explainable, which makes comparing interpretable models (Logistic Regression, Decision Tree) against black-box ones (ensembles, neural networks) meaningful.

<h4> Checking the structure of the data as well as data types and count of nulls. This helps in knowing which columns contain missing values and how to impute them. </h4>

In [15]:
print("no of rows is:", df_raw.shape[0])
print("no of columns is:", df_raw.shape[1])
# print("data types:")
# print(df_raw.dtypes)
print(df_raw.dtypes.value_counts())
print()
df_raw.info()

no of rows is: 10302
no of columns is: 27
str        16
int64       8
float64     3
Name: count, dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 10302 entries, 0 to 10301
Data columns (total 27 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          10302 non-null  int64  
 1   KIDSDRIV    10302 non-null  int64  
 2   BIRTH       10302 non-null  str    
 3   AGE         10295 non-null  float64
 4   HOMEKIDS    10302 non-null  int64  
 5   YOJ         9754 non-null   float64
 6   INCOME      9732 non-null   str    
 7   PARENT1     10302 non-null  str    
 8   HOME_VAL    9727 non-null   str    
 9   MSTATUS     10302 non-null  str    
 10  GENDER      10302 non-null  str    
 11  EDUCATION   10302 non-null  str    
 12  OCCUPATION  9637 non-null   str    
 13  TRAVTIME    10302 non-null  int64  
 14  CAR_USE     10302 non-null  str    
 15  BLUEBOOK    10302 non-null  str    
 16  TIF         10302 non-null  int64  
 17  CAR_TYP

**Findings:** 10,302 rows × 27 columns. pandas reads 16 columns as text (`str`), but 5 of them (`INCOME`, `HOME_VAL`, `BLUEBOOK`, `OLDCLAIM`, `CLM_AMT`) are really currency amounts stored as strings such as `"$67,349"`. Six columns have fewer than 10,302 non-null values, so they contain missing values.

<h1>Data Understanding</h1>
<h2> Lets explain the data more and the meaning behind each column </h2>

In [16]:
TARGET = "CLAIM_FLAG"

# (semantic type, description) for every column
column_info = {
    "ID":         ("Identifier",         "Policy / customer identifier"),
    "KIDSDRIV":   ("Numeric - discrete", "Number of children who drive"),
    "BIRTH":      ("Date (text)",        "Date of birth (DDMONYY)"),
    "AGE":        ("Numeric - discrete", "Age of the driver (years)"),
    "HOMEKIDS":   ("Numeric - discrete", "Number of children at home"),
    "YOJ":        ("Numeric - discrete", "Years on current job"),
    "INCOME":     ("Currency (text)",    "Annual income ($)"),
    "PARENT1":    ("Binary",             "Single parent (Yes/No)"),
    "HOME_VAL":   ("Currency (text)",    "Home value ($), 0 = no home owned"),
    "MSTATUS":    ("Binary",             "Married (Yes/No)"),
    "GENDER":     ("Binary",             "Gender (M/F)"),
    "EDUCATION":  ("Ordinal",            "Highest education level"),
    "OCCUPATION": ("Nominal",            "Job category"),
    "TRAVTIME":   ("Numeric - discrete", "Travel distance/time to work"),
    "CAR_USE":    ("Binary",             "Vehicle use (Private/Commercial)"),
    "BLUEBOOK":   ("Currency (text)",    "Market value of the vehicle ($)"),
    "TIF":        ("Numeric - discrete", "Time in force: years as a customer"),
    "CAR_TYPE":   ("Nominal",            "Vehicle type"),
    "RED_CAR":    ("Binary",             "Vehicle is red (yes/no)"), #its funny but its a common belief that red cars get more accidents
    "OLDCLAIM":   ("Currency (text)",    "Total claim payouts in the past 5 years ($)"),
    "CLM_FREQ":   ("Numeric - discrete", "Number of claims in the past 5 years"),
    "REVOKED":    ("Binary",             "Licence revoked in the past 7 years (Yes/No)"),
    "MVR_PTS":    ("Numeric - discrete", "Motor-vehicle-record (traffic violation) points"),
    "CLM_AMT":    ("Currency (text)",    "Amount of the current claim ($)"),
    "CAR_AGE":    ("Numeric - discrete", "Age of the vehicle (years)"),
    "CLAIM_FLAG": ("Target (binary)",    "1 = customer filed a claim, 0 = no claim"),
    "URBANICITY": ("Binary",             "Home area (Urban/Rural)"),
}

data_dictionary = pd.DataFrame({
    "feature_type":   {c: v[0] for c, v in column_info.items()},
    "description":    {c: v[1] for c, v in column_info.items()},
    "pandas_dtype":   df_raw.dtypes.astype(str),
    "n_unique":       df_raw.nunique(),
    "n_missing":      df_raw.isna().sum(),
    "missing_%":      (df_raw.isna().mean() * 100).round(2),
    "example_values": {c: list(df_raw[c].dropna().unique()[:3]) for c in df_raw.columns},
}).loc[df_raw.columns]

print(data_dictionary["feature_type"].value_counts(), "\n")
data_dictionary

feature_type
Numeric - discrete    9
Binary                7
Currency (text)       5
Nominal               2
Identifier            1
Date (text)           1
Ordinal               1
Target (binary)       1
Name: count, dtype: int64 



,feature_type,description,pandas_dtype,n_unique,n_missing,missing_%,example_values
ID,Identifier,Policy / customer identifier,int64,8753,0,0.00,"[63581743, 132761049, 921317019]"
KIDSDRIV,Numeric - discrete,Number of children who drive,int64,5,0,0.00,"[0, 1, 2]"
BIRTH,Date (text),Date of birth (DDMONYY),str,6560,0,0.00,"[16MAR39, 21JAN56, 18NOV51]"
AGE,Numeric - discrete,Age of the driver (years),float64,61,7,0.07,"[60.0, 43.0, 48.0]"
HOMEKIDS,Numeric - discrete,Number of children at home,int64,6,0,0.00,"[0, 1, 2]"
YOJ,Numeric - discrete,Years on current job,float64,21,548,5.32,"[11.0, 10.0, 14.0]"
INCOME,Currency (text),Annual income ($),str,8151,570,5.53,"[$67,349, $91,449, $52,881]"
PARENT1,Binary,Single parent (Yes/No),str,2,0,0.00,"[No, Yes]"
HOME_VAL,Currency (text),"Home value ($), 0 = no home owned",str,6334,575,5.58,"[$0, $257,252, $124,191]"
MSTATUS,Binary,Married (Yes/No),str,2,0,0.00,"[z_No, Yes]"


<h1> There are 9 numeric, 5 fields are currency but stored as string, 7 binary but include categorical binary, 1 is ordinal which is the Education (order matters, PHD > Masters), 2 are nominal, 1 date and 1 ID </h1> 
<h3> each group require their own preprocssing currency has to become numbers categorical features has to be incoded </h3>

<h1> Now lets explore the target variables and class balance </h1>
<h4> this will be done by counting how many customers are in each class and calculate the ratio. Also, we will calculate the accuracy of a model that wil always predict the higher class, That is the minimum to beat </h4>
<h5> <i> why would we have a model that can't even have higher accuracy a model that always predict one class (the higher class) </i> </h5>

In [17]:
target_counts=df_raw[TARGET].value_counts()
target_table = pd.DataFrame({
    "count": target_counts,
    "percentage": (target_counts / len(df_raw) * 100).round(2),
})
target_table.index = ["0 - No claim", "1 - Claim"]
print(target_table, "\n")
#len(df_raw) returns the number of rows/ instances
#imbalance ratio = 
print(f"Imbalance ratio (majority : minority) = {target_counts.max() / target_counts.min():.2f} : 1")
print(f"Accuracy of always predicting 'No claim' = {target_counts.max() / len(df_raw):.2%}")

              count  percentage
0 - No claim   7556       73.34
1 - Claim      2746       26.66 

Imbalance ratio (majority : minority) = 2.75 : 1
Accuracy of always predicting 'No claim' = 73.34%


<h1> 73.34% Of customers didnot claim and 26.66% did claim, there is a clear imbalance ratio of 2.75:1</h1>
<h4> predicting no claim for everyone already gives 73.34% accuracy, meaning that accuracy is not reliable here and we shall evaluate models with other techniques like recall percision f1-score and ROC-AUC</h4>
<h4> it's also importnat to stratify the train-test split to keep this ratio </h4>

<h4> Now lets compute the missing values and compare claim rate against rows where a feature is missing and where it is present, if they are equal this means the missingnes is not in a critical feature (feature related to the target) and could be imputed. </h4>

In [18]:
missing_table = (pd.DataFrame({"n_missing": df_raw.isna().sum(),
                               "missing_%": (df_raw.isna().mean() * 100).round(2)})
                 .query("n_missing > 0")
                 .sort_values("missing_%", ascending=False))
# print(missing_table, "\n")
total_missing_cells_pct=df_raw.isna().sum().sum()/df_raw.size*100
rows_with_missing_pct = df_raw.isna().any(axis=1).mean() * 100
print(f"Missing cells overall      : {total_missing_cells_pct:.2f}% of all values")
print(f"Rows with >=1 missing value: {df_raw.isna().any(axis=1).sum()} ({rows_with_missing_pct:.2f}% of rows)\n")
#axis =1 because we want to see if a row contain a null not if a column (axis =0)
#now checking if misisng values related to target
missing_vs_target = pd.DataFrame({
    "claim_rate_when_missing_%": {c: df_raw.loc[df_raw[c].isna(), TARGET].mean() * 100 for c in missing_table.index},
    "claim_rate_when_present_%": {c: df_raw.loc[df_raw[c].notna(), TARGET].mean() * 100 for c in missing_table.index},
}).round(2)
missing_vs_target

Missing cells overall      : 1.08% of all values
Rows with >=1 missing value: 2645 (25.67% of rows)



,claim_rate_when_missing_%,claim_rate_when_present_%
OCCUPATION,26.17,26.69
CAR_AGE,27.39,26.61
HOME_VAL,25.74,26.71
INCOME,24.56,26.78
YOJ,26.82,26.65
AGE,85.71,26.61


**Findings:** 6 columns have missing values: `OCCUPATION` (6.46%), `CAR_AGE` (6.20%), `HOME_VAL` (5.58%), `INCOME` (5.53%), `YOJ` (5.32%) and `AGE` (0.07%, only 7 rows). Only 1.08% of all cells are missing, but they are spread out, so **25.67% of rows have at least one missing value**. Dropping those rows would throw away a quarter of the data, so **imputation** is the better choice. For the five main columns, the claim rate is almost the same whether the value is missing or present (e.g. INCOME 24.6% vs 26.8%), which suggests the missingness is not strongly related to the target. (The `AGE` difference is based on only 7 rows and is not meaningful.)

<h1> now lets check other data quality problems that are not just missing values</h1>
<h4> currency for example is stored as text and there is inconsistent category labels, repeated id's,
duplicated rows, unreal values, a column that leaks the value </h4>

In [19]:
# 1) Currency stored as text
money_cols = [c for c, v in column_info.items() if v[0] == "Currency (text)"]
print("Currency columns stored as text:", money_cols)
print(df_raw[money_cols].head(3), "\n") #sample values

# 2) Category labels with a 'z_' prefix
z_prefixed = {c: sorted(v for v in df_raw[c].dropna().unique() if str(v).startswith("z_"))
              for c in df_raw.select_dtypes(exclude="number").columns}
print("Categories with a 'z_' prefix:")
for c, v in z_prefixed.items():
    if v:
        print(f"  {c}: {v}")

# 3) Duplicates and repeated IDs
n_exact_dupes = df_raw.duplicated().sum()
dup_id_rows = df_raw[df_raw["ID"].duplicated(keep=False)] #keep = false to return all duplicates of an instancem important to know why id's repeated
same_person = dup_id_rows.groupby("ID")["BIRTH"].nunique().eq(1).sum()
print(f"\nExact duplicate rows           : {n_exact_dupes}")
print(f"IDs that appear more than once : {dup_id_rows['ID'].nunique()} (covering {len(dup_id_rows)} rows)")
print(f"  - same birth date (likely same customer, different car/policy): {same_person}")
print(f"  - different birth dates (different people sharing an ID)      : {dup_id_rows['ID'].nunique() - same_person}")

#Impossible values
print("\nCAR_AGE values below 1 year:")
print(df_raw.loc[df_raw["CAR_AGE"] < 1, "CAR_AGE"].value_counts())

# Target leakage, meaning a feature leaks the answer of the target in this case CLM_AMT
claim_amount = df_raw["CLM_AMT"].str.replace(r"[$,]", "", regex=True).astype(float)
print("\nLeakage check  CLM_AMT > 0 versus CLAIM_FLAG:")
#frequency table to show
pd.crosstab(claim_amount > 0, df_raw[TARGET], rownames=["CLM_AMT > 0"], colnames=["CLAIM_FLAG"])

Currency columns stored as text: ['INCOME', 'HOME_VAL', 'BLUEBOOK', 'OLDCLAIM', 'CLM_AMT']
    INCOME  HOME_VAL BLUEBOOK OLDCLAIM CLM_AMT
0  $67,349        $0  $14,230   $4,461      $0
1  $91,449  $257,252  $14,940       $0      $0
2  $52,881        $0  $21,970       $0      $0 

Categories with a 'z_' prefix:
  MSTATUS: ['z_No']
  GENDER: ['z_F']
  EDUCATION: ['z_High School']
  OCCUPATION: ['z_Blue Collar']
  CAR_TYPE: ['z_SUV']
  URBANICITY: ['z_Highly Rural/ Rural']

Exact duplicate rows           : 1
IDs that appear more than once : 1332 (covering 2881 rows)
  - same birth date (likely same customer, different car/policy): 503
  - different birth dates (different people sharing an ID)      : 829

CAR_AGE values below 1 year:
CAR_AGE
 0.0    4
-3.0    1
Name: count, dtype: int64

Leakage check  CLM_AMT > 0 versus CLAIM_FLAG:


CLAIM_FLAG,0,1
CLM_AMT > 0,,
False,7556,0
True,0,2746


**Findings:**
1. **Currency as text:** `INCOME`, `HOME_VAL`, `BLUEBOOK`, `OLDCLAIM` and `CLM_AMT` contain `$` and `,`, so they must be converted to numbers.
2. **`z_` prefix:** six columns have one category written with a `z_` prefix (`z_No`, `z_F`, `z_High School`, `z_Blue Collar`, `z_SUV`, `z_Highly Rural/ Rural`). This is a leftover from how the data was exported, and the prefix must be removed.
3. **Duplicates:** only **1 exact duplicate row**, which will be removed. However, **1,332 IDs appear more than once (2,881 rows)**. 829 of these groups have different birth dates, so they are different people who happen to share an ID. 503 share a birth date, which suggests the same customer with a different car or policy. Because the rows are genuinely different records, they are kept, and `ID` is dropped as a feature because an identifier carries no predictive meaning.
4. **Impossible values:** one car has `CAR_AGE = -3`, which is impossible and will be treated as missing. Four cars have `CAR_AGE = 0`, which is plausible (a brand-new car) and kept.
5. **Target leakage:** `CLM_AMT > 0` matches `CLAIM_FLAG = 1` **perfectly** (0 exceptions out of 10,302 rows). The claim amount is only known *after* a claim happens, so a model using it would be "cheating" and useless in practice. **`CLM_AMT` must be removed.**
6. **Redundancy:** `BIRTH` carries the same information as `AGE`, so it will be dropped.

<h1> now lets check the outliers and distribution </h1>

In [20]:
#lets first convert currency to number temporarily (will do that in details later)
def parse_money(series):
    """Convert text like '$67,349' to the number 67349.0 (NaN stays NaN)."""
    return series.str.replace(r"[$,]", "", regex=True).astype(float)

df_num_view = df_raw.copy()
for c in money_cols:
    df_num_view[c] = parse_money(df_num_view[c])

numeric_cols = [c for c, v in column_info.items()
                if v[0] in ("Numeric - discrete", "Currency (text)")]

rows = {}
for c in numeric_cols:
    x = df_num_view[c].dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((x < low) | (x > high)).sum()
    rows[c] = {"min": x.min(), "median": x.median(), "max": x.max(),
               "lower_fence": low, "upper_fence": high,
               "n_outliers": n_out, "outliers_%": round(n_out / len(x) * 100, 2),
               "skewness": round(x.skew(), 2), "zeros_%": round((x == 0).mean() * 100, 2)}

outlier_table = pd.DataFrame(rows).T.sort_values("outliers_%", ascending=False)
outlier_table

,min,median,max,lower_fence,upper_fence,n_outliers,outliers_%,skewness,zeros_%
CLM_AMT,0.0,0.0,123247.0,-1717.125,2861.875,1974.0,19.16,9.30,73.34
KIDSDRIV,0.0,0.0,4.0,0.000,0.000,1233.0,11.97,3.34,88.03
HOMEKIDS,0.0,0.0,5.0,-1.500,2.500,1075.0,10.43,1.34,64.98
YOJ,0.0,11.0,23.0,3.000,19.000,837.0,8.58,-1.20,8.27
OLDCLAIM,0.0,0.0,57037.0,-6971.250,11618.750,837.0,8.12,3.12,61.08
INCOME,0.0,53529.0,367030.0,-60289.000,174039.000,301.0,3.09,1.16,8.19
MVR_PTS,0.0,1.0,13.0,-4.500,7.500,203.0,1.97,1.34,45.21
TIF,1.0,4.0,25.0,-8.000,16.000,194.0,1.88,0.90,0.00
BLUEBOOK,1500.0,14400.0,69740.0,-8335.000,38425.000,116.0,1.13,0.77,0.00
TRAVTIME,5.0,33.0,142.0,-11.000,77.000,80.0,0.78,0.44,0.00


<h1>now lets make a copy of the data (to clean it and keep original refernce for comparison and future need) </h1>

In [21]:
df = df_raw.copy()
cleaning_log = []          # (action, count) pairs for the final summary

n_before = len(df) #no of rows before
df = df.drop_duplicates().reset_index(drop=True) #dropping duplicates in the copy and resetting the index (to start from 0 again)
cleaning_log.append(("Removed exact duplicate rows", n_before - len(df)))

print(f"Rows before: {n_before} | after: {len(df)} | removed: {n_before - len(df)}")

Rows before: 10302 | after: 10301 | removed: 1


### Now lets covert currency to numbers actually via regex using a previous helper function

In [22]:
CURRENCY_COLS = ["INCOME", "HOME_VAL", "BLUEBOOK", "OLDCLAIM", "CLM_AMT"]

for c in CURRENCY_COLS:
    df[c] = parse_money(df[c])
cleaning_log.append(("Converted currency text to numbers", len(CURRENCY_COLS)))

print(df[CURRENCY_COLS].dtypes, "\n")
df[CURRENCY_COLS].head(3)

INCOME      float64
HOME_VAL    float64
BLUEBOOK    float64
OLDCLAIM    float64
CLM_AMT     float64
dtype: object 



,INCOME,HOME_VAL,BLUEBOOK,OLDCLAIM,CLM_AMT
0,67349.0,0.0,14230.0,4461.0,0.0
1,91449.0,257252.0,14940.0,0.0,0.0
2,52881.0,0.0,21970.0,0.0,0.0


### Fixing category labels

In [23]:
text_cols = df.select_dtypes(include=["object", "string"]).columns.drop("BIRTH")

for c in text_cols:
    df[c] = df[c].str.strip().str.replace(r"^z_", "", regex=True)

df["URBANICITY"] = df["URBANICITY"].replace({"Highly Urban/ Urban": "Urban",
                                             "Highly Rural/ Rural": "Rural"})
df["RED_CAR"] = df["RED_CAR"].str.capitalize()
cleaning_log.append(("Removed 'z_' prefix / tidied labels", len(text_cols)))

for c in text_cols:
    print(f"{c:<11}: {sorted(df[c].dropna().unique())}")

PARENT1    : ['No', 'Yes']
MSTATUS    : ['No', 'Yes']
GENDER     : ['F', 'M']
EDUCATION  : ['<High School', 'Bachelors', 'High School', 'Masters', 'PhD']
OCCUPATION : ['Blue Collar', 'Clerical', 'Doctor', 'Home Maker', 'Lawyer', 'Manager', 'Professional', 'Student']
CAR_USE    : ['Commercial', 'Private']
CAR_TYPE   : ['Minivan', 'Panel Truck', 'Pickup', 'SUV', 'Sports Car', 'Van']
RED_CAR    : ['No', 'Yes']
REVOKED    : ['No', 'Yes']
URBANICITY : ['Rural', 'Urban']


### Before dropping the column of Birth (redundant with Age) can we use it to impute missing Age values ?

In [24]:
birth_date = pd.to_datetime(df["BIRTH"].str[:5] + "19" + df["BIRTH"].str[-2:], format="%d%b%Y")

REFERENCE_DATE = pd.Timestamp("1999-11-17")
had_birthday = ((birth_date.dt.month < REFERENCE_DATE.month) |
                ((birth_date.dt.month == REFERENCE_DATE.month) &
                 (birth_date.dt.day <= REFERENCE_DATE.day)))
age_from_birth = REFERENCE_DATE.year - birth_date.dt.year - (~had_birthday).astype(int)

known = df["AGE"].notna()
match = (age_from_birth[known] == df.loc[known, "AGE"]).mean()
print(f"AGE recomputed from BIRTH matches the recorded AGE for {match:.2%} of rows\n")

print("Rows where AGE is missing:")
pd.DataFrame({"BIRTH": df.loc[~known, "BIRTH"], "age_from_birth": age_from_birth[~known]})

AGE recomputed from BIRTH matches the recorded AGE for 99.88% of rows

Rows where AGE is missing:


,BIRTH,age_from_birth
295,18FEB86,13
1317,21NOV83,15
1651,18OCT84,15
3749,17FEB86,13
4365,15FEB84,15
4554,17AUG85,14
5227,03DEC87,11


**Findings:** the recomputed age matches the recorded `AGE` for **99.88%** of rows, so `BIRTH` adds no new information and is **dropped as redundant**. For the 7 rows with missing `AGE`, `BIRTH` would give ages of **11–15**. That is below driving age and below the dataset minimum of 16, so those birth dates are unreliable, which probably explains why their `AGE` is missing. We therefore **do not** fill `AGE` from `BIRTH`. It stays missing and will be imputed in preprocessing.

<h1> checking for impossible values like car age < 0 </h1>

In [25]:
# Impossible values -> missing (they will be imputed later)
n_neg = (df["CAR_AGE"] < 0).sum()
df.loc[df["CAR_AGE"] < 0, "CAR_AGE"] = np.nan
cleaning_log.append(("CAR_AGE < 0 set to missing", int(n_neg)))

job_start_age = df["AGE"] - df["YOJ"]
impossible_yoj = job_start_age < 10
df.loc[impossible_yoj, "YOJ"] = np.nan
cleaning_log.append(("YOJ implying a job start before age 10 set to missing", int(impossible_yoj.sum())))

print(f"CAR_AGE < 0 set to NaN            : {n_neg}")
print(f"YOJ set to NaN (job start age <10): {impossible_yoj.sum()}\n")

# Consistency checks
print("Past claim payouts (OLDCLAIM > 0) vs past claim count (CLM_FREQ > 0):")
print(pd.crosstab(df["OLDCLAIM"] > 0, df["CLM_FREQ"] > 0,
                  rownames=["OLDCLAIM > 0"], colnames=["CLM_FREQ > 0"]), "\n")
print(f"Single parents with no children at home: "
      f"{((df['PARENT1'] == 'Yes') & (df['HOMEKIDS'] == 0)).sum()}\n")
print("Customers with INCOME = 0, by occupation:")
print(df.loc[df["INCOME"] == 0, "OCCUPATION"].value_counts(dropna=False))

CAR_AGE < 0 set to NaN            : 1
YOJ set to NaN (job start age <10): 13

Past claim payouts (OLDCLAIM > 0) vs past claim count (CLM_FREQ > 0):
CLM_FREQ > 0  False  True 
OLDCLAIM > 0              
False          6291      0
True              0   4010 

Single parents with no children at home: 0

Customers with INCOME = 0, by occupation:
OCCUPATION
Student         402
Home Maker      378
Lawyer            8
Doctor            6
Professional      2
Manager           1
Name: count, dtype: int64


**Findings:** 1 negative `CAR_AGE` and 13 impossible `YOJ` values (job start before age 10) were set to missing. The consistency checks pass: `OLDCLAIM > 0` exactly when `CLM_FREQ > 0` (no contradictions), and every single parent has children at home. Zero income belongs almost entirely to Students (402) and Home Makers (378), so `INCOME = 0` is a real value, not a missing one. The 17 high-earning occupations with zero income (e.g. 8 lawyers) are rare but not impossible (between jobs, for example), so they are kept.

<h1> dropping unusable columns and the leakage column </h1>

In [26]:
customer_id = df["ID"].copy()      # kept aside for the train/test split step

DROP_COLS = ["ID", "BIRTH", "CLM_AMT"]
df = df.drop(columns=DROP_COLS)
cleaning_log.append(("Dropped columns (ID, BIRTH, CLM_AMT)", len(DROP_COLS)))

print("Remaining columns:", df.shape[1])
print(list(df.columns))

Remaining columns: 24
['KIDSDRIV', 'AGE', 'HOMEKIDS', 'YOJ', 'INCOME', 'PARENT1', 'HOME_VAL', 'MSTATUS', 'GENDER', 'EDUCATION', 'OCCUPATION', 'TRAVTIME', 'CAR_USE', 'BLUEBOOK', 'TIF', 'CAR_TYPE', 'RED_CAR', 'OLDCLAIM', 'CLM_FREQ', 'REVOKED', 'MVR_PTS', 'CAR_AGE', 'CLAIM_FLAG', 'URBANICITY']


### 3.7 Feature groups and data types
Every feature is assigned to exactly one group, and each group gets its own preprocessing later:
- **Numeric (13):** imputation + scaling
- **Categorical – binary (7):** 0/1 encoding
- **Categorical – ordinal (1):** `EDUCATION`, encoded in its natural order
- **Categorical – nominal (2):** `OCCUPATION`, `CAR_TYPE`, one-hot encoding

Categorical columns are stored as pandas `category`, and `EDUCATION` as an *ordered* category, so the data type itself records the feature type. An `assert` checks that no column is missing from a group or assigned twice.

In [27]:
NUMERIC_FEATURES = ["KIDSDRIV", "AGE", "HOMEKIDS", "YOJ", "INCOME", "HOME_VAL", "TRAVTIME",
                    "BLUEBOOK", "TIF", "OLDCLAIM", "CLM_FREQ", "MVR_PTS", "CAR_AGE"]
BINARY_FEATURES  = ["PARENT1", "MSTATUS", "GENDER", "CAR_USE", "RED_CAR", "REVOKED", "URBANICITY"]
ORDINAL_FEATURES = ["EDUCATION"]
NOMINAL_FEATURES = ["OCCUPATION", "CAR_TYPE"]
EDUCATION_ORDER  = ["<High School", "High School", "Bachelors", "Masters", "PhD"]

CATEGORICAL_FEATURES = BINARY_FEATURES + ORDINAL_FEATURES + NOMINAL_FEATURES
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Safety checks: every column is in exactly one group, and the education order covers all values
assert set(ALL_FEATURES) | {TARGET} == set(df.columns)
assert len(ALL_FEATURES) == len(set(ALL_FEATURES))
assert set(df["EDUCATION"].unique()) == set(EDUCATION_ORDER)

for c in CATEGORICAL_FEATURES:
    df[c] = df[c].astype("category")
df["EDUCATION"] = pd.Categorical(df["EDUCATION"], categories=EDUCATION_ORDER, ordered=True)

print(f"Numeric: {len(NUMERIC_FEATURES)} | Binary: {len(BINARY_FEATURES)} | "
      f"Ordinal: {len(ORDINAL_FEATURES)} | Nominal: {len(NOMINAL_FEATURES)}\n")
print(df.dtypes)

Numeric: 13 | Binary: 7 | Ordinal: 1 | Nominal: 2

KIDSDRIV         int64
AGE            float64
HOMEKIDS         int64
YOJ            float64
INCOME         float64
PARENT1       category
HOME_VAL       float64
MSTATUS       category
GENDER        category
EDUCATION     category
OCCUPATION    category
TRAVTIME         int64
CAR_USE       category
BLUEBOOK       float64
TIF              int64
CAR_TYPE      category
RED_CAR       category
OLDCLAIM       float64
CLM_FREQ         int64
REVOKED       category
MVR_PTS          int64
CAR_AGE        float64
CLAIM_FLAG       int64
URBANICITY    category
dtype: object
